# Bagré Sentinel-2 segmentation: guided Colab runner

Before starting: export the two GeoTIFFs with `gee/01_export_bagre.js`, place them in `bagre-segmentation/raw`, and upload this repository to `bagre-segmentation/bagre-sentinel2-segmentation` in Google Drive. Select a GPU runtime. This notebook does not contain pre-filled results.

Run through the data summary first. Inspect the spatial split and class counts, then enable training in the Random Forest cell. Use a new RUN_NAME for changed inputs or settings.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Select Runtime > Change runtime type > GPU before the real training run.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os, subprocess, sys

PROJECT = Path('/content/drive/MyDrive/bagre-sentinel2-segmentation')
REPO = PROJECT
RAW = PROJECT / 'raw'
RUN_NAME = 'run_02'
RUN_DIR = PROJECT / 'runs' / RUN_NAME
PROCESSED = RUN_DIR / 'processed'
RF_DIR = RUN_DIR / 'rf'
UNET_DIR = RUN_DIR / 'unet'
COMPARISON = RUN_DIR / 'comparison'
IMAGE = RAW / 'bagre_s2_2021_dryseason.tif'
LABELS = RAW / 'bagre_worldcover_2021.tif'

required = [REPO / 'pyproject.toml', REPO / 'configs/bagre.yaml', REPO / 'scripts/prepare_data.py', REPO / 'scripts/train_rf.py', REPO / 'scripts/train_unet.py', REPO / 'scripts/compare_models.py', IMAGE, LABELS]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
print('Working directory:', Path.cwd())

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest'], check=True)

## 1. Prepare aligned patches and spatial splits

The preparation step fails if the rasters do not share exactly the same CRS, transform, height, and width. It also warns about missing classes. Do not ignore those warnings.

In [ ]:
assert IMAGE.exists(), f'Missing {IMAGE}'
assert LABELS.exists(), f'Missing {LABELS}'
subprocess.run([
    sys.executable, 'scripts/prepare_data.py',
    '--image', str(IMAGE), '--labels', str(LABELS),
    '--output-dir', str(PROCESSED)
], check=True)

In [ ]:
import json
from IPython.display import Image as DisplayImage, display
with open(PROCESSED / 'metadata.json') as handle:
    metadata = json.load(handle)
print('Patch counts:', metadata['split_counts'])
print('Class support:', metadata['class_support_pixels'])
display(DisplayImage(filename=str(PROCESSED / 'data_summary.png')))
display(DisplayImage(filename=str(PROCESSED / 'spatial_split.png')))

## 2. Train and evaluate the Random Forest baseline

The primary baseline uses only B2, B3, B4, and B8, matching U-Net's spectral inputs. Training pixels are sampled by class from the training region only.

In [ ]:
RUN_TRAINING = True  # Enable after reviewing the prepared data.
if not RUN_TRAINING:
    raise RuntimeError('Review the data summary before enabling training in the RF cell.')
subprocess.run([
    sys.executable, 'scripts/train_rf.py',
    '--data-dir', str(PROCESSED), '--output-dir', str(RF_DIR)
], check=True)

## 3. Train U-Net and evaluate the best checkpoint

The best checkpoint is selected using validation mIoU. Test evaluation runs after training. Checkpoints are saved to Drive every epoch.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before training U-Net.')
if not RUN_TRAINING:
    raise RuntimeError('Review the data summary before enabling training in the RF cell.')
RESUME_UNET = False
checkpoint = UNET_DIR / 'last.pt'
if RESUME_UNET and not checkpoint.is_file():
    raise FileNotFoundError(checkpoint)
if checkpoint.is_file() and not RESUME_UNET:
    raise RuntimeError('Set RESUME_UNET for the same experiment or choose a fresh RUN_NAME.')
resume_args = ['--resume', str(checkpoint)] if RESUME_UNET else []
subprocess.run([
    sys.executable, 'scripts/train_unet.py',
    '--data-dir', str(PROCESSED), '--output-dir', str(UNET_DIR)
] + resume_args, check=True)

After a disconnect, remount Drive and rerun setup. If the processed data and settings are unchanged, keep the same RUN_NAME, set RUN_TRAINING and RESUME_UNET to True, and run the U-Net cell. Use a fresh RUN_NAME for changed inputs or settings.

## 4. Compare models and inspect evidence

In [ ]:
subprocess.run([
    sys.executable, 'scripts/compare_models.py',
    '--data-dir', str(PROCESSED), '--unet-dir', str(UNET_DIR),
    '--rf-dir', str(RF_DIR), '--output-dir', str(COMPARISON)
], check=True)
display(DisplayImage(filename=str(COMPARISON / 'prediction_examples.png')))
display(DisplayImage(filename=str(COMPARISON / 'unet_training_curves.png')))

In [ ]:
import pandas as pd
results = pd.read_csv(COMPARISON / 'results.csv')
display(results)
print('Interpretation: agreement with ESA WorldCover, not field-validated accuracy.')
print('Next: complete docs/MODEL_CARD_TEMPLATE.md and docs/INTERVIEW_CHECKLIST.md from these artifacts.')

In [ ]:
columns = [
    column for column in results.columns
    if column == "model"
    or column.startswith(("iou_", "support_", "predictions_"))
]
display(results[columns].T)